# 05 - Predator-Prey: the predator that wanders off

Same game as notebook 03: 3 predators chase 1 faster prey; the prey is frozen for the first 0.5M training steps, then learns.

Look at the **predators' reward** again: every step, **+10 for each predator touching the prey - and every predator gets it**.
* A predator gets the same reward whether it caught the prey itself, helped corner it, or was somewhere else entirely.
* Predators are **not** penalised for leaving the arena (only the prey is).

So each predator has to work out from a *shared* number whether *its own* action helped. This is the **credit-assignment problem**.
If it goes wrong, an agent can stop contributing and still collect the team's reward - a **lazy agent**.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import *
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90
COL = {"maddpg": "tab:blue", "facvdn": "tab:orange", "facqmix": "tab:green"}

## 1. MADDPG: one predator leaves

Trained MADDPG predators (2M steps) in three episodes, seen from above. Circles = start, crosses = end, dashed square = arena.

In [ ]:
maddpg = tag_seed_runs("maddpg")
pol_m = snapshot(maddpg[0], 2_000_000)
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, ep in zip(ax, (27, 29, 0)):
    plot_tag_paths(pol_m, seed=ep, ax=a, title=f"MADDPG, episode {ep}")
ax[0].legend(fontsize=8)
plt.tight_layout()

Two predators chase the prey; the third runs off the map **away from the prey** and never comes back.

In [ ]:
show_video(video(pol_m, "tag", {}, seed=27, title="MADDPG predators (2M steps)"))

Why does MADDPG's critic not prevent this? Each predator's critic Q_i(s, a_1, a_2, a_3) predicts the **team** reward from everything.
When the other two predators are doing the catching, the team value hardly depends on the third predator's action, so its actor
gets almost no gradient - and there is no cost to drifting.

## 2. Factored critics: VDN and QMIX for continuous actions

Value decomposition gives each predator its **own** value, learned from the shared reward (FACMAC-style: keep the DDPG actors, change only the critic):

* each predator *i* has a utility **Q_i(o_i, a_i)** - from its own observation and its own action only;
* **VDN:** Q_tot = Q_1 + Q_2 + Q_3
* **QMIX:** Q_tot = f_s(Q_1, Q_2, Q_3) - a mixing network whose weights (always >= 0) come from small networks that look at the global state s.

Only Q_tot is trained on the team reward. Each actor climbs **dQ_tot / da_i**; for VDN that is exactly **dQ_i / da_i** - a predator is
pushed by the value of *its own* action, not by whatever the team happened to get.
The prey is unchanged (MADDPG critic), so the only difference to section 1 is the predators' critic.

| | MADDPG critic (per predator) | factored critic (team) |
|---|---|---|
| input | all observations + all actions | own observation + own action |
| trained on | team reward | team reward, through Q_tot |
| actor follows | dQ_i(s, a_1..a_3)/da_i | dQ_tot/da_i (VDN: dQ_i(o_i, a_i)/da_i) |

In [ ]:
vdn, qmix = tag_seed_runs("facvdn"), tag_seed_runs("facqmix")
print("MADDPG"); describe_networks(maddpg[0])
print("\nFAC-VDN"); describe_networks(vdn[0])
mix = torch.load(qmix[0]["dir"] / "final.pt", weights_only=False)["mixer"]
print(f"\nFAC-QMIX: same networks + a mixing network (hypernetworks of the state): {sum(v.numel() for v in mix.values()):,} parameters")

## 3. Same episodes, factored critics

In [ ]:
pol_v, pol_q = snapshot(vdn[0], 2_000_000), snapshot(qmix[0], 2_000_000)
fig, ax = plt.subplots(3, 3, figsize=(14, 14))
for r, (pol, name) in enumerate([(pol_m, "MADDPG"), (pol_v, "FAC-VDN"), (pol_q, "FAC-QMIX")]):
    for c, ep in enumerate((27, 29, 0)):
        plot_tag_paths(pol, seed=ep, ax=ax[r, c], title=f"{name}, episode {ep}")
ax[0, 0].legend(fontsize=8)
plt.tight_layout()

In [ ]:
show_video(video(pol_m, "tag", {}, seed=27, title="MADDPG") + video(pol_v, "tag", {}, seed=27, title="FAC-VDN")
           + video(pol_q, "tag", {}, seed=27, title="FAC-QMIX"))

Every predator now heads for the prey.

## Takeaways
* **A shared reward hides who did the work.** MADDPG predators developed a predator that leaves the arena and stops chasing - in all
  3 seeds, against every prey - while still receiving the full team reward.
* **Value decomposition** (VDN's sum, QMIX's monotonic mix) gives each agent its own value Q_i(o_i, a_i); with it, the wandering predator
  disappears (3/3 seeds each).
* **Honest caveats:**
  * It is not a "lazy agent" in the strict sense: the wanderer still makes some touches (the least-involved predator's share of touches is
    similar for all three methods).
  * Better behaviour is not automatically more catches: against the same prey, the factored predators catch more at the end of
    training, but MADDPG's caught more at 1M steps (try `tag_algo_crossplay(1_000_000)`).
  * QMIX needed an extra stabiliser (twin critics).
  * While the prey was frozen and catching was easy, no method had a wandering predator: credit assignment bites when the reward becomes rare.